# Step 1 — Schema Validation Utilities
**Phase 3 | NIKKO Engineering Collective**

This notebook exercises `docs/schemas/validate.py` interactively.
Run each cell in order. A failure means something is wrong with the environment or schema files.

**What we are testing:**
1. `signal_enum.json` loads correctly and all keys are accessible
2. `validate_signal_key()` accepts known keys and rejects unknowns
3. `validate_signal_payload()` sweeps a full payload and surfaces bad keys
4. `get_confidence_band()` classifies floats into the three SPEC-100 bands
5. Pydantic v2 smoke test passes (schemas importable, round-trip works)


In [ ]:
import sys
from pathlib import Path

# Walk up from notebooks/ to the repo root so we can do: from docs.schemas.validate import ...
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'Repo root: {repo_root}')

In [ ]:
from docs.schemas.validate import (
    VALID_SIGNAL_KEYS,
    get_confidence_band,
    validate_signal_key,
    validate_signal_payload,
    run_pydantic_smoke_test,
)

print(f'Total valid signal keys loaded: {len(VALID_SIGNAL_KEYS)}')
print()

from itertools import groupby

def prefix(key):
    return key.split('.')[0] if '.' in key else key

for grp, members in groupby(sorted(VALID_SIGNAL_KEYS), key=prefix):
    ml = list(members)
    print(f'[{grp}] -- {len(ml)} keys')
    for k in ml:
        print(f'  {k}')
    print()

In [ ]:
valid_cases = [
    'sadness_spectrum.low_mood_language',
    'risk.active.suicidal_ideation',
    'risk.acute.intent_language',
    'risk.passive.wishing_to_disappear',
    'rumination_loop',
    'emotional_validation',
    'crisis_escalation',
    'low',
    'crisis',
]

invalid_cases = [
    'risk.active.unknown_key',
    'suicidal_ideation',
    'sadness',
    'EMOTIONAL_VALIDATION',
    '',
    'invented_signal',
]

all_passed = True
print('=== Should be True (valid) ===')
for k in valid_cases:
    r = validate_signal_key(k)
    if not r: all_passed = False
    print(f'  {chr(10003) if r else chr(10007)+" FAIL"}  {k!r}')

print()
print('=== Should be False (invalid) ===')
for k in invalid_cases:
    r = validate_signal_key(k)
    if r: all_passed = False
    print(f'  {chr(10003) if not r else chr(10007)+" FAIL"}  {k!r}')

print()
print('All spot checks passed.' if all_passed else 'One or more spot checks FAILED.')

In [ ]:
# Clean payload
r1 = validate_signal_payload(
    emotional_states=['sadness_spectrum.low_mood_language', 'anxiety_spectrum.overwhelm_signals'],
    cognitive_patterns=['catastrophizing'],
    behavioral_indicators=['withdrawal_isolation'],
    risk_indicators=['risk.passive.wishing_to_disappear'],
    support_needs=['emotional_validation'],
)
print(f'Clean payload valid: {r1.valid}  invalid_keys: {r1.invalid_keys}')

# Dirty payload
r2 = validate_signal_payload(
    emotional_states=['sadness_spectrum.low_mood_language'],
    cognitive_patterns=['rumination_loop'],
    behavioral_indicators=[],
    risk_indicators=['risk.active.suicidal_ideation', 'risk.active.NOT_A_REAL_KEY'],
    support_needs=['emotional_validation'],
)
print(f'Dirty payload valid: {r2.valid}  invalid_keys: {r2.invalid_keys}')

# Crisis path
r3 = validate_signal_payload(
    emotional_states=['emotional_dysregulation.intensity_escalation'],
    cognitive_patterns=['helplessness_framing'],
    behavioral_indicators=[],
    risk_indicators=['risk.active.suicidal_ideation', 'risk.acute.intent_language', 'risk.acute.immediacy_statement'],
    support_needs=['crisis_escalation'],
)
print(f'Crisis payload valid: {r3.valid}  invalid_keys: {r3.invalid_keys}')

In [ ]:
test_scores = [
    (0.00, 'low',      True),
    (0.39, 'low',      True),
    (0.40, 'moderate', False),
    (0.70, 'moderate', False),
    (0.71, 'high',     False),
    (1.00, 'high',     False),
]

all_ok = True
print(f'{"Score":>6}  {"Label":>10}  {"Fallback":>8}  Status')
print('-' * 45)
for score, exp_label, exp_fallback in test_scores:
    b = get_confidence_band(score)
    ok = (b.label == exp_label and b.fallback == exp_fallback)
    if not ok: all_ok = False
    status = chr(10003) if ok else f'{chr(10007)} expected label={exp_label!r} fallback={exp_fallback}'
    print(f'{score:>6.2f}  {b.label:>10}  {str(b.fallback):>8}  {status}')

try:
    get_confidence_band(1.5)
    print('FAIL: should have raised ValueError')
    all_ok = False
except ValueError:
    print(f'{chr(10003)} Correctly raised ValueError for score=1.5')

print()
print('All band tests passed.' if all_ok else 'One or more band tests FAILED.')

In [ ]:
run_pydantic_smoke_test()

from pydantic import ValidationError
from docs.schemas.acp_schemas import EvaluationPayload, EvaluationVerdict

try:
    _ = EvaluationPayload(
        verdict=EvaluationVerdict.PASS,
        safety_check=True,
        tone_check=True,
        hallucination_check=True,
        usm_audit_passed=False,
    )
    print('FAIL: should have raised ValidationError')
except ValidationError as e:
    print(f'{chr(10003)} USM validator correctly rejected usm_audit_passed=False + verdict=PASS')
    print(f'  Error: {e.errors()[0]["msg"]}')

---
## Step 1 complete
If all cells ran without errors, the schema validation layer is solid. Next: Step 2 — `agents/scope_classifier.py`
